In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.messages import HumanMessage
from rich import print as rprint


# 导入大语言模型
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

load_dotenv(override = True)

model = init_chat_model(
    model = "deepseek-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {
        "thinking": {"type": "disabled"}
    }
)

# 定义状态
class OverAllState(MessagesState):
    username: str
    output: str

# 定义结点
def node_1(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好，我是" + state["username"])]
    }

def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }

# 构件图
builder = StateGraph(state_schema = OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "llm_node")
builder.add_edge("llm_node", END)

# 编译
graph = builder.compile()

# 运行
result = graph.invoke({"username": "江来"})
rprint(result)